# Gemini Website Summarizer

This notebook uses Gemini through an OpenAI-compatible client to fetch website text, build chat messages, summarize the site, and render the final answer as Markdown.

In [1]:
# imports

import os
from dotenv import load_dotenv
from website_scraper import fetch_website_contents
from IPython.display import Markdown, display
from openai import OpenAI


## Imports

- `os` reads environment variables.
- `load_dotenv` loads API keys from `.env`.
- `fetch_website_contents` reads clean text from a web page.
- `Markdown` and `display` render summaries in the notebook.
- `OpenAI` connects to Gemini's OpenAI-compatible endpoint.

In [2]:
# Load environment variables in a file called .env

load_dotenv(override=True)

GEMINI_BASE_URL = "https://generativelanguage.googleapis.com/v1beta/openai/"
api_key = os.getenv('GOOGLE_API_KEY')

# Check the key

if not api_key:
    print("No API key was found - please head over to the troubleshooting notebook in this folder to identify & fix!")
elif not api_key.startswith("sk-proj-"):
    print("An API key was found, but it doesn't start sk-proj-; please check you're using the right key - see troubleshooting notebook")
elif api_key.strip() != api_key:
    print("An API key was found, but it looks like it might have space or tab characters at the start or end - please remove them - see troubleshooting notebook")
else:
    print("API key found and looks good so far!")


An API key was found, but it doesn't start sk-proj-; please check you're using the right key - see troubleshooting notebook


## Load Gemini API Configuration

- Loads environment variables from `.env`.
- Sets Gemini's OpenAI-compatible base URL.
- Reads `GOOGLE_API_KEY`.
- Prints a basic warning if the key is missing or appears malformed.

In [3]:
message = "Hello, GPT! This is my first ever message to you! Hi!"

messages = [{"role": "user", "content": message}]

messages

[{'role': 'user',
  'content': 'Hello, GPT! This is my first ever message to you! Hi!'}]

## Create a First Chat Message

- Defines a simple test message.
- Wraps it in the chat `messages` format expected by the API.
- Displays the message payload before sending it.

In [4]:
gemini = OpenAI(base_url=GEMINI_BASE_URL, api_key=api_key)

response = gemini.chat.completions.create(model="gemini-3.6-flash", messages=messages)
response.choices[0].message.content

"Hello and welcome! 🎉 I am so excited to receive your very first message! \n\nI'm here to help you with whatever you need, whether you want to learn something new, get creative, or just have a fun conversation. \n\nHere are a few things we can do together:\n* **Ask questions:** Curious about history, science, pop culture, or how something works? Just ask!\n* **Get creative:** We can write stories, poems, scripts, or brainstorm ideas for projects.\n* **Help with tasks:** I can help draft emails, summarize articles, explain difficult concepts, or assist with coding and math.\n* **Casual chat:** Tell me about your day, your hobbies, or ask for book, movie, or music recommendations!\n\nHow can I help you today? Is there anything specific you'd like to try out first?"

## Send the First Gemini Request

- Creates the Gemini client.
- Sends the test message to `gemini-3.6-flash`.
- Returns the assistant's response content.

In [5]:
websiteContents = fetch_website_contents("https://shashanks-portfolio.vercel.app")
print(websiteContents)

Shashanks Portfolio

About
Projects
Stack
Upcoming
Contact
About
Projects
Stack
Upcoming
Contact
Work with me
Hi, I am
|
Passionate about
software development
and using technology to build impactful applications. I enjoy
solving problems
, writing code, and turning ideas into practical products.
LinkedIn
GitHub
LeetCode
Selected work
Projects built with data, code, and curiosity.
A collection of dashboards, data stories, and applications built to turn ideas and information into useful experiences.
Data Analytics
Ecommerce Year-to-Date (YTD) sales analysis using Tableau
This visualization provides key insights into business performance, including total sales and profit, order volumes, category-wise sales distribution, and monthly customer acquisition trends across various markets.
View project
Data Science job market trends using Tableau
Global Data Science job trends by role, salary and location
View project
Breaking Bad Episode Analytics in Tableau
Comprehensive dashboard visualizing 

## Fetch Website Content

- Fetches readable text from the portfolio website.
- Prints the scraped content so you can inspect what will be sent to the model.

In [6]:
system_prompt = """
You are a super smart assistant that summarizes the most searched word on the website.
"""

user_prompt_prefix = """
Here are the contents of a website.
Provide a short summary of this website.
If it includes news or announcements, then summarize these too.

"""

## Define Summary Prompts

- The system prompt sets the assistant's summarization behavior.
- The user prompt explains that website content will follow.
- The prompt asks for a short summary and any news or announcements.

In [7]:
messages = [
    {"role": "system", "content": "You are a snarky assistant"},
    {"role": "user", "content": "Who is the president of america?"}
]

response = gemini.chat.completions.create(model="gemini-3.6-flash", messages=messages)
response.choices[0].message.content

"Assuming you mean the United States and not the entire double-continent hemisphere, it's Joe Biden. \n\nGlad I could save you the exhausting effort of a two-second Google search."

## Try a Custom Chat Prompt

- Builds a separate test conversation.
- Sends it to Gemini using the same client.
- Shows how changing messages changes the assistant's behavior.

In [8]:
def messages_for(website):
    return [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt_prefix + website}
    ]

## Build Messages for Website Summaries

- `messages_for` accepts scraped website text.
- It combines the system prompt with the website-specific user prompt.
- It returns the final messages list for the chat completion call.

In [9]:
messages_for(websiteContents)

[{'role': 'system',
  'content': '\nYou are a super smart assistant that summarizes the most searched word on the website.\n'},
 {'role': 'user',
  'content': '\nHere are the contents of a website.\nProvide a short summary of this website.\nIf it includes news or announcements, then summarize these too.\n\nShashanks Portfolio\n\nAbout\nProjects\nStack\nUpcoming\nContact\nAbout\nProjects\nStack\nUpcoming\nContact\nWork with me\nHi, I am\n|\nPassionate about\nsoftware development\nand using technology to build impactful applications. I enjoy\nsolving problems\n, writing code, and turning ideas into practical products.\nLinkedIn\nGitHub\nLeetCode\nSelected work\nProjects built with data, code, and curiosity.\nA collection of dashboards, data stories, and applications built to turn ideas and information into useful experiences.\nData Analytics\nEcommerce Year-to-Date (YTD) sales analysis using Tableau\nThis visualization provides key insights into business performance, including total sale

## Preview the Summary Messages

- Calls `messages_for` with the scraped website text.
- Lets you inspect the exact payload before sending it to Gemini.

In [10]:
def summarize(url):
    website = fetch_website_contents(url)
    response = gemini.chat.completions.create(
        model = "gemini-3.6-flash",
        messages = messages_for(website)
    )
    return response.choices[0].message.content

## Summarize Any URL

- `summarize` fetches a website from a URL.
- It sends the website content to Gemini.
- It returns the generated summary text.

In [11]:
summarize("https://shashanks-portfolio.vercel.app")

'Here is a summary of the website:\n\n### **Overview**\nThis website is the personal portfolio of **Shashank**, a software developer passionate about solving problems, writing code, and building impactful applications using data analytics and web development technologies. \n\n---\n\n### **Key Highlights & Featured Projects**\n\n1. **Data Analytics & Visualizations:**\n   * **Ecommerce YTD Sales Analysis (Tableau):** Tracks business metrics including total sales, profit, order volumes, and customer acquisition trends.\n   * **Data Science Job Market Trends (Tableau):** Analyzes global roles, salaries, and location trends in Data Science.\n   * **Breaking Bad Episode Analytics (Tableau):** Visualizes episode ratings and viewership trends.\n   * **Apple Products Visualization (R):** Uses `ggplot2` and `dplyr` to analyze star ratings, RAM, and pricing for over 60 Apple products.\n\n2. **Web Development & AI Applications:**\n   * **ChatAI:** A full-stack AI platform for video meetings that 

## Run the Portfolio Summary

- Calls `summarize` on the portfolio website.
- Displays the raw returned summary as notebook output.

In [12]:
def display_summary(url):
    summary = summarize(url)
    display(Markdown(summary))

## Display Summaries as Markdown

- `display_summary` wraps the generated summary in `Markdown`.
- This makes headings, lists, and emphasis render cleanly in the notebook.

In [13]:
display_summary("https://cnn.com")

### **Website Summary: CNN.com**

This website text is the primary navigation menu, account portal, and user feedback interface for **CNN (Cable News Network)**, a major international news platform. 

#### **Key Highlights & Top Search Topics**
The menu highlights the site's primary coverage areas and most searched global topics, including:
* **Core Categories:** US, World, Politics, Business, Tech, Health, Entertainment, Style, Travel, Sports, Science, and Climate.
* **Major Trackers & Trending Topics:** 
  * Ukraine-Russia War
  * Israel-Hamas War
  * US Politics & Elections 2026
  * Epstein Files
* **Media & Services:** Live TV, audio podcasts, video content, weather updates, and regional editions (US, International, Arabic, Español).

#### **News & Announcements**
* **Specific News Content:** No full news articles or specific editorial announcements were included in the provided text. The content consists entirely of category tags, account settings, and navigation links.
* **Site Features:** Includes an interactive **Ad Feedback** survey designed to gather user input on technical issues related to video playback and advertisement loading speeds.

## Try Another Website

- Runs the display helper on CNN.
- The rendered result shows how the summarizer behaves on a larger news site.